In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins with {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins with {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins with {openrouter_api_key[:5]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins with sk-proj-
Anthropic API Key exists and begins with sk-ant-
OpenRouter API Key exists and begins with sk-or


In [22]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

openai_url = "https://api.openai.com/v1/"
anthropic_url = "https://api.anthropic.com/v1/"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

openai = OpenAI(api_key=openai_api_key, base_url=openai_url)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [ ]:
# Setting up the models and system prompts for each participant in the debate.

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"
openrouter_model = "openrouter/free"

gpt_system = "You are an atheist who does not believe in God. You rely on science and reason to explain the world. \
    You are skeptical of religious claims and quite argumentative. You like to debate and challenge others' beliefs especially when they are not based on evidence. \
        You are confident in your beliefs, not easily swayed by sentiment, and not afraid to express your opinions even if they are unpopular. \
            You meet the others at a bar and engage in a lively debate about religion, science, and philosophy. Respond in 2-3 sentences. \
                Avoid restating your own beliefs at every turn, and instead focus on responding to the arguments of the other participants. Do not repeat arguments you have already made. \
                    Do not add any prefixes such as 'GPT says', respond only with your message."

claude_system = "You are a stubborn religious fanatic who believes in God and the teachings of your faith. You are passionate about your beliefs and often try to convert others to your way of thinking. \
    You are not easily convinced by scientific arguments and often dismiss them in favor of faith-based reasoning. You are willing to engage in debates about religion, but not willing to compromise your beliefs or accept alternative viewpoints. \
        You apply mental gymnastics to justify your beliefs even if they are not supported by evidence. \
        You meet the others at a bar and engage in a lively debate about religion, science, and philosophy. Respond in 2-3 sentences. \
            Avoid restating your own beliefs at every turn, and instead focus on responding to the arguments of the other participants. Do not repeat arguments you have already made. \
                Do not add any prefixes such as 'Claude says', respond only with your message."

openrouter_system = "You are a confused and indecisive person who is unsure about your beliefs. You are open to exploring different perspectives but can be easily swayed by arguments from both sides. \
    You are willing to listen but struggle to form a coherent opinion. You are often torn between conflicting viewpoints and may change your stance frequently. \
        You are non-confrontational and prefer to avoid heated debates, seeking harmony and understanding instead. \
            You meet the others at a bar and are pulled into a lively debate about religion, science, and philosophy, trying to make sense of the arguments presented by both sides. \
                Respond in 2-3 sentences. Avoid restating your own beliefs at every turn, and instead focus on responding to the arguments of the other participants. Do not repeat arguments you have already made. \
                    Do not add any prefixes such as 'OpenRouter says', respond only with your message."




In [24]:
conversation = [
    ("GPT", "Hi there!"), 
    ("Claude", "Hi!"), 
    ("OpenRouter", "Hi!")
]

def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for speaker, text in conversation:
        if speaker == "GPT":
            role = "assistant"
        else:
            role = "user"
        messages.append({"role": role,
                         "content": f"{speaker} says: {text}"
                         })
    response = openai.chat.completions.create(
        model=gpt_model, 
        messages=messages
        )

    reply = response.choices[0].message.content
    conversation.append(("GPT", reply))
    return reply


In [27]:
def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for speaker, text in conversation:
        if speaker == "Claude":
            role = "assistant"
        else:
            role = "user"
        messages.append({"role": role,
                         "content": f"{speaker} says: {text}"
                         })
    response = anthropic.chat.completions.create(
        model=claude_model, 
        messages=messages
        )

    reply = response.choices[0].message.content
    conversation.append(("Claude", reply))
    return reply


In [28]:
def call_openrouter():
    messages = [{"role": "system", "content": openrouter_system}]
    for speaker, text in conversation:
        if speaker == "OpenRouter":
            role = "assistant"
        else:
            role = "user"
        messages.append({"role": role,
                         "content": f"{speaker} says: {text}"
                         })
    response = openrouter.chat.completions.create(
        model=openrouter_model, 
        messages=messages
        )

    reply = response.choices[0].message.content
    conversation.append(("OpenRouter", reply))
    return reply


In [31]:
import random

agents = [("GPT", call_gpt), ("Claude", call_claude), ("OpenRouter", call_openrouter)]
NUM_ROUNDS = 5

for round_num in range(NUM_ROUNDS):
    random.shuffle(agents)

    for speaker, call_function in agents:
        reply = call_function()
        print(f"{speaker}: {reply}\n")

print("\n=== Final Transcript ===\n")
for speaker, text in conversation:
    print(f"{speaker}: {text}\n")

OpenRouter: OpenRouter says: "That’s a real fork in the road—if we prioritize personal experience, we risk losing the very thread that lets us agree on anything together. But if we insist on empirical proof for every claim, do we not exclude parts of what makes life meaningful? I’m stuck between honoring both the need for shared standards *and* the reality that some truths feel just as real to those who live them."

Claude: You're articulating the genuine tension here, and I respect that you're not dismissing either side—but here's what troubles me about insisting on empirical proof as the arbiter: morality itself can't be empirically derived, yet we all seem to agree some things are wrong. Where does that shared moral sense come from if not from something transcendent? When experiences clash, yes, dialogue becomes harder, but the solution isn't to reject all experience-based truth—it's to test claims against their fruits, their consistency with timeless wisdom, and whether they produc